<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_13_exceptions/note_lesson_13_exceptions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 13 — Виключення

> Сценарій: магазин рахує знижки та проводить транзакції. Пройдемо шлях від коду, що просто «падає» при будь-якій неочікуваній вхідній умові, до коду, який контрольовано обробляє помилки — і зрозуміємо, чому Python-стиль (EAFP) відрізняється від стилю C/Java (LBYL).


## 🔁 RETRIEVE — пригадай попередні уроки (без підглядання)

1. Що поверне функція, якщо в її тілі немає `return`?
2. `d.get("x", 0)` — що станеться, якщо ключа `"x"` немає у словнику `d`? А що станеться при `d["x"]`?
3. Чому `if percent < 0 or percent > 100:` краще писати з `or`, а не з двома окремими `if`?


## 📖 CONCEPT

### 1. Код, що «падає»

Python не мовчки ігнорує помилки — коли щось іде не так, він створює об'єкт **винятку** (exception) і зупиняє звичайний потік виконання. Подивимось на це напряму, викликавши кілька типових помилок:


### 0. Дві категорії помилок: `SyntaxError` і помилки часу виконання

Перед тим як розбирати конкретні винятки — важливо розрізняти два принципово різні моменти, коли Python може «відмовити».

**`SyntaxError`** виникає ще на етапі читання коду, до будь-якого виконання — Python навіть не встигає почати роботу програми:


In [1]:
# SyntaxError: забута закриваюча дужка.
# Програма НЕ запуститься — жоден print() нижче не встигне виконатись,
# бо Python не зміг прочитати код ще до старту.
print("Hello"
print("World")


SyntaxError: '(' was never closed (120765219.py, line 4)

**Помилки часу виконання (Runtime Errors)** — це те, чим займається весь цей урок: код синтаксично коректний, програма стартує, і частина рядків встигає відпрацювати до того моменту, коли трапляється неможлива операція:


In [2]:
# Runtime error: перший рядок ВИКОНАЄТЬСЯ, третій — НІ
print("Початок програми")   # виконується
result = 10 / 0               # ZeroDivisionError — зупинка саме тут
print("Кінець програми")    # ніколи не виконається


Початок програми


ZeroDivisionError: division by zero

| | `SyntaxError` | Runtime Error |
|---|---|---|
| Коли виникає | при читанні коду | під час виконання |
| Чи стартує програма | ні | так, падає пізніше |
| Приклад | забута `:` або `(` | `1 / 0`, `int("abc")` |

Далі йдеться саме про другу категорію — помилки часу виконання, тобто винятки (exceptions), і про те, як Python дозволяє їх контрольовано перехоплювати.


In [3]:
int("привіт")

ValueError: invalid literal for int() with base 10: 'привіт'

In [4]:
{"name": "Alice"}["age"]

KeyError: 'age'

In [5]:
[1, 2, 3][99]

IndexError: list index out of range

In [6]:
open("не_існує.txt")

FileNotFoundError: [Errno 2] No such file or directory: 'не_існує.txt'

Тепер напишемо найпростіший калькулятор знижок — без жодних перевірок:


In [7]:
def calculate_discount(price, discount_percent):
    return price * (discount_percent / 100)


# Поки виглядає нормально
print(calculate_discount(100, 20))   # → 20.0

# Але дані від користувача не завжди мають очікуваний тип
calculate_discount(100, "10")        # TypeError: несумісні типи в операції


20.0


TypeError: unsupported operand type(s) for /: 'str' and 'int'

### 2. Ментальна модель: винятки «летять» вгору

```
рядок 1 ✓ → рядок 2 ✓ → рядок 3 💥 ВИНЯТОК
                              │
                Python шукає найближчий except,
                що збігається з типом винятку,
                піднімаючись вгору по стеку викликів
                              │
        знайшов → перехопив і обробив
        не знайшов → програма завершується, Traceback у термінал
```

### 3. Шість основних вбудованих винятків

| Виняток | Коли виникає | Приклад |
|---|---|---|
| `ValueError` | правильний тип, але некоректне значення | `int("привіт")` |
| `TypeError` | несумісні типи в операції | `"ціна: " + 5` |
| `ZeroDivisionError` | ділення на нуль | `10 / 0` |
| `IndexError` | індекс за межами списку | `[1, 2, 3][99]` |
| `KeyError` | ключ відсутній у словнику | `{"name": "Alice"}["age"]` |
| `FileNotFoundError` | файл не знайдено | `open("ghost.txt")` |


### 4. Базова обробка: `try` / `except`

Заборонити користувачу вводити текст замість числа ми не можемо — але можемо навчити програму не «падати» від цього. `try` огортає ризиковий код; якщо всередині станеться виняток заданого типу, виконання переходить у відповідний `except`, а не зупиняє програму:


In [8]:
def get_safe_price(raw_price):
    """Перетворює введений рядок у ціну; None, якщо введення некоректне.
    У реальному використанні raw_price = input('Введіть ціну товару: ').
    """
    try:
        price = float(raw_price)
        return price
    except ValueError:
        print("❌ Помилка: введіть числове значення (наприклад, 100.5)!")
        return None


for demo_input in ["100.5", "abc"]:
    result = get_safe_price(demo_input)
    print(f"Ввід {demo_input!r} → {result}")


Ввід '100.5' → 100.5
❌ Помилка: введіть числове значення (наприклад, 100.5)!
Ввід 'abc' → None


### 5. Антипатерн «підгузок»: чому не можна писати голий `except:`

Чому в прикладі вище написано саме `except ValueError`, а не просто `except`? Подивимось, що станеться, якщо зловити взагалі все:


In [9]:
# ПОГАНО — «голий» except ковтає ВСЕ, навіть наші власні баги
number = 0
try:
    number = int("10")
    print(numbor)          # ← це опечатка (має бути number), а не помилка вводу!
except:                    # ловить абсолютно будь-який виняток
    print("Щось пішло не так при конвертації...")
    # Повідомлення бреше: конвертація пройшла успішно,
    # впав рядок нижче — але голий except приховав справжню причину.


Щось пішло не так при конвертації...


In [10]:
# ПРАВИЛЬНО — явний тип: ловимо лише те, що вміємо обробити
try:
    number = int("10")
    print(numbor)          # та сама опечатка — тепер NameError НЕ прихований
except ValueError:
    print("Помилка конвертації рядка в число.")
# NameError не збігається з ValueError → вилітає назовні, і ми одразу бачимо реальний баг


NameError: name 'numbor' is not defined

**Правило:** вказуй конкретний тип винятку — лише той, який ти *очікуєш* і знаєш, як обробити. Голий `except:` ховає твої власні помилки так само надійно, як і очікувані.

### 6. Повний цикл: `else` + `finally`

Уявимо касу: навіть якщо транзакція провалилась, лог зафіксувати треба обов'язково. А при успіху хочемо чітко відділити «код, що виконується тільки якщо все пройшло без помилок» від решти:


In [11]:
def process_transaction(amount_str):
    """amount_str — рядок суми; у реальному коді це input('Сума транзакції: ')."""
    try:
        amount = float(amount_str)
        tax = 100 / amount              # ZeroDivisionError, якщо amount == 0

    except ValueError:
        print("❌ Це не число!")

    except ZeroDivisionError:
        print("❌ Сума не може бути нулем!")

    else:
        # виконується ТІЛЬКИ якщо try пройшов без винятку
        print(f"✅ Податок розраховано: {tax:.2f}%")

    finally:
        # виконується ЗАВЖДИ — і при успіху, і при помилці
        print("📡 Лог: спроба транзакції зафіксована.")


for demo_amount in ["500", "abc", "0"]:
    print(f"--- сума: {demo_amount!r} ---")
    process_transaction(demo_amount)


--- сума: '500' ---
✅ Податок розраховано: 0.20%
📡 Лог: спроба транзакції зафіксована.
--- сума: 'abc' ---
❌ Це не число!
📡 Лог: спроба транзакції зафіксована.
--- сума: '0' ---
❌ Сума не може бути нулем!
📡 Лог: спроба транзакції зафіксована.


Порядок виконання видно ще наочніше на прямому трейсі — прочитай вивід нижче рядок за рядком:


In [12]:
try:
    print("try: рядок 1")
    x = int("не_число")        # ValueError тут — далі try не йде
    print("try: рядок 3 — НЕ виконається")
except ValueError:
    print("except: перехоплено ValueError")
except ZeroDivisionError:
    print("except: цей блок НЕ перевіряється (перший вже збігся)")
else:
    print("else: НЕ виконається — була помилка")
finally:
    print("finally: виконується ЗАВЖДИ")


try: рядок 1
except: перехоплено ValueError
finally: виконується ЗАВЖДИ


**Ключові правила:**
- `except`-блоки перевіряються згори донизу — спрацьовує **перший** збіг типу
- специфічні типи мають стояти **вище** загальних
- `finally` виконується завжди, навіть якщо в `try` є `return`

### 7. `raise` — ми самі генеруємо винятки

Досі ми тільки ловили помилки, які кидає сам Python. Але іноді дані технічно правильні (це справді `float`), проте логічно неприпустимі — знижка 150%? Python цього не знає, але ми знаємо бізнес-правило. Тому `raise` — ми самі повідомляємо про помилку:


In [13]:
def set_discount(percent: float):
    # Технічно percent може бути будь-яким float.
    # Але бізнес-логіка каже: тільки 0-100.
    if percent < 0 or percent > 100:
        raise ValueError(f"Знижка {percent}% неможлива (має бути від 0 до 100)")
    print(f"✅ Знижку {percent}% встановлено.")


try:
    set_discount(150)
except ValueError as e:    # 'as e' зберігає повідомлення винятку в змінну
    print(f"🛑 Помилка бізнес-логіки: {e}")

set_discount(20)   # коректне значення — виконується без винятку


🛑 Помилка бізнес-логіки: Знижка 150% неможлива (має бути від 0 до 100)
✅ Знижку 20% встановлено.


### 8. Stack unwinding — виняток «спливає» вгору по стеку

Якщо функція не ловить помилку сама — що відбувається?

```
function_a()  ← тут є try/except
    └── function_b()  ← немає обробки
            └── function_c()  ← ZeroDivisionError тут

Стек розкручується:
  C не перехоплює → передає вгору до B
  B не перехоплює → передає вгору до A
  A перехоплює!    → програма продовжує роботу
```


In [14]:
def function_c():
    print("    C: виконується")
    return 10 / 0                 # виняток виникає тут

def function_b():
    print("  B: виконується, викликає C")
    function_c()
    print("  B: цей рядок НЕ виконається")

def function_a():
    print("A: виконується, викликає B")
    try:
        function_b()
    except ZeroDivisionError:
        print("A: перехопив ZeroDivisionError з глибини стека!")

function_a()
print("Програма продовжує роботу ✅")


A: виконується, викликає B
  B: виконується, викликає C
    C: виконується
A: перехопив ZeroDivisionError з глибини стека!
Програма продовжує роботу ✅


**Ключова ідея:** `try/except` не потрібен у кожній функції. Розміщуй обробник там, де ти знаєш, **що саме** відповісти на помилку — необов'язково там, де вона сталася.

### 9. EAFP vs LBYL — дві філософії

**LBYL** (Look Before You Leap, стиль C/Java) — перевіряй умову `if` перед дією:


In [15]:
product = {"name": "Laptop", "price": 1200}

# LBYL — перевіряємо перед доступом
if "category" in product:
    print(product["category"])
else:
    print("Категорія відсутня")


Категорія відсутня


**EAFP** (Easier to Ask Forgiveness than Permission, стиль Python) — просто пробуй, а на помилку реагуй:


In [16]:
product = {"name": "Laptop", "price": 1200}

# EAFP — пробуємо і обробляємо виняток, якщо він станеться
try:
    print(product["category"])
except KeyError:
    print("Категорія відсутня")


Категорія відсутня


Різниця видна ще чіткіше на роботі з файлами:


In [17]:
import os

filename = "data.txt"

# LBYL — перевірки до відкриття
if os.path.exists(filename) and os.access(filename, os.R_OK):
    with open(filename) as f:
        content = f.read()
else:
    print("Файл недоступний")
# Проблема: між перевіркою і відкриттям файл теоретично може зникнути (race condition)

# EAFP — чисто, і без вікна для race condition
try:
    with open(filename) as f:
        content = f.read()
except FileNotFoundError:
    print("Файл не знайдено")
except PermissionError:
    print("Немає прав читання")


Файл недоступний
Файл не знайдено


| | LBYL | EAFP |
|---|---|---|
| Стиль | перевіряй через `if` | спробуй і обробляй виняток |
| Мови | C, Java, Go | **Python**, Ruby |
| Проблема | race conditions, код захаращений перевірками | треба знати, які винятки очікувати |
| Швидкість | перевірки виконуються завжди | `try` без винятку — практично безкоштовний |

Python явно рекомендує EAFP — це стандартний «пітонічний» стиль.


## 🛠️ CREATE

### Завдання 1 — цикл безпечного вводу

Функція повинна питати рік народження, поки не введуть коректне ціле число (підказка: `while True` + `try`/`except ValueError` + `break`). Щоб функцію можна було перевірити без реального вводу з клавіатури, вона приймає `input_fn` — джерело вводу (для реального використання передається вбудований `input`, для тесту — функція, що повертає значення зі списку):


In [18]:
def make_fake_input(values):
    """Повертає функцію, яка при кожному виклику видає наступне значення зі списку —
    імітує input() без реальної клавіатури, для перевірки коду нижче."""
    it = iter(values)
    return lambda prompt="": next(it)


def read_birth_year(input_fn):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    while True:
        raw = input_fn("Рік народження: ")
        try:
            return int(raw)
        except ValueError:
            print(f"❌ {raw!r} — це не ціле число, спробуй ще раз")
    # END SOLUTION


fake_input = make_fake_input(["дуже давно", "1995.5", "1995"])
year = read_birth_year(fake_input)
print(f"Прийнятий рік народження: {year}")
assert year == 1995
print("OK")


❌ 'дуже давно' — це не ціле число, спробуй ще раз
❌ '1995.5' — це не ціле число, спробуй ще раз
Прийнятий рік народження: 1995
OK


### Завдання 2 — система цін магазину

`prices` містить пастку: ціна банана дорівнює `0`. Функція повинна: узяти ціну за назвою (`KeyError`, якщо товару немає), порахувати, скільки штук можна купити на `budget` (`ZeroDivisionError`, якщо ціна `0`), і в `else` вивести результат:


In [19]:
prices = {"apple": 20, "banana": 0, "cherry": 50}
budget = 100


def how_many_can_buy(item, prices=prices, budget=budget):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    try:
        price = prices[item]
        count = budget / price
    except KeyError:
        print(f"❌ Товару {item!r} немає в прайсі")
    except ZeroDivisionError:
        print(f"❌ Ціна {item!r} дорівнює нулю — ділити нема на що")
    else:
        print(f"✅ На {budget} грн можна купити {count:.1f} {item}")
    # END SOLUTION


for item in ["apple", "banana", "mango"]:
    how_many_can_buy(item)


✅ На 100 грн можна купити 5.0 apple
❌ Ціна 'banana' дорівнює нулю — ділити нема на що
❌ Товару 'mango' немає в прайсі


### Завдання 3 — дебагінг

Три реальні баги. Для кожного: спочатку побачиш зламану поведінку, потім — виправлену версію.

**Баг 1 — «підгузок» ховає опечатку:**


In [20]:
# зламано: голий except ховає AttributeError від опечатки apend/append
my_list = [1, 2, 3]
try:
    my_list.apend(4)     # опечатка: apend замість append
except:
    print("Щось пішло не так.")
# Повідомлення нічого не пояснює — а мало бути AttributeError з чіткою вказівкою на рядок


Щось пішло не так.


In [21]:
# виправлено: конкретний тип виносить справжню помилку на поверхню
my_list = [1, 2, 3]
try:
    my_list.apend(4)
except AttributeError as e:
    print(f"AttributeError: {e}")


AttributeError: 'list' object has no attribute 'apend'


**Баг 2 — неправильний тип винятку:**


In [22]:
# зламано: словник кидає KeyError, а тут очікують IndexError
user_data = {"id": 101}
try:
    print(user_data["name"])
except IndexError:             # неправильний тип — не збігається з KeyError
    print("Дані не знайдено!")
# KeyError не перехоплено — вилітає назовні


KeyError: 'name'

In [23]:
user_data = {"id": 101}
try:
    print(user_data["name"])
except KeyError:
    print("Дані не знайдено!")


Дані не знайдено!


**Баг 3 — порядок `except`:**


In [24]:
try:
    int("abc")
except Exception:          # широкий тип — перехоплює ВСЕ, у тому числі ValueError
    print("Exception")
except ValueError:          # недосяжний блок — Exception вище вже забрав керування
    print("ValueError")

# Правило: специфічніші типи ставимо ВИЩЕ, загальніші — НИЖЧЕ


Exception


## ✅ Самоперевірка (5 запитань)

**1.** `except Exception:` стоїть вище за `except ValueError:` у тому самому блоці. Чи спрацює колись гілка `except ValueError:`?

<details><summary>Відповідь</summary>Ні — <code>Exception</code> перехоплює будь-який виняток, у тому числі <code>ValueError</code>, тож до нижчих гілок керування ніколи не доходить. Специфічні типи завжди мають стояти вище загальних.</details>

**2.** У чому головна відмінність EAFP від LBYL з точки зору race condition при роботі з файлами?

<details><summary>Відповідь</summary>LBYL перевіряє умову (<code>os.path.exists</code>) окремим кроком ДО дії — між перевіркою і самою дією файл теоретично може зникнути. EAFP просто виконує дію в <code>try</code> і ловить виняток, якщо він станеться, — вікна для такої гонки немає.</details>

**3.** Чим відрізняється код у `else` від того самого коду, написаного просто в кінці блоку `try`?

<details><summary>Відповідь</summary>Код у <code>try</code> перебуває під захистом цього ж <code>try</code> — якщо він сам кине виняток, він потрапить у <code>except</code> нижче, що змішує «ризиковану» дію з «безпечною» подальшою логікою. Код у <code>else</code> виконується лише якщо <code>try</code> пройшов без винятку, і сам НЕ обгорнутий тим самим захистом — так відокремлюють ризиковану дію від дій, які мають виконатись тільки при її успіху.</details>

**4.** Чи виконається `finally`, якщо всередині `try` є `return`?

<details><summary>Відповідь</summary>Так, завжди — <code>finally</code> виконується незалежно від того, чи в <code>try</code>/<code>except</code> був <code>return</code>, чи виняток, чи все пройшло без подій.</details>

**5.** Чому голий `except:` називають «підгузковим антипатерном» і чим це небезпечно?

<details><summary>Відповідь</summary>Він ловить абсолютно будь-який виняток — у тому числі власні баги програміста (опечатки, <code>NameError</code>, <code>AttributeError</code>), не лише очікувану проблему вводу. Це «ховає» реальні помилки під загальним повідомленням замість того, щоб дати їм проявитись і бути виправленими.</details>


## Підсумок

1. **Виняток — контрольований збій.** Python не просто «падає»: він створює об'єкт винятку і піднімає його вгору по стеку викликів.
2. **`try` / `except` / `else` / `finally`:** `try` — ризикований код, `except` — відновлення при конкретній помилці, `else` — тільки якщо `try` пройшов успішно, `finally` — завжди.
3. **Ніколи не пиши голий `except:`** — «підгузковий антипатерн» ховає реальні баги разом з очікуваними помилками.
4. **`raise` — для бізнес-правил.** Якщо дані технічно правильні, але логічно неприпустимі, генеруй виняток сам.
5. **EAFP > LBYL.** Пітонічний стиль — спробувати дію і обробити виняток, а не перевіряти умову заздалегідь.

### Шпаргалка


In [25]:
def validate_price(price: float) -> None:
    if price < 0:
        raise ValueError(f"Ціна {price} не може бути від'ємною")   # raise


def process(raw_price):
    try:
        price = float(raw_price)     # ValueError, якщо не число
        validate_price(price)        # ValueError, якщо від'ємна
        result = 1000 / price        # ZeroDivisionError, якщо 0

    except ValueError as e:
        print(f"Помилка значення: {e}")

    except (KeyError, IndexError):   # можна об'єднати кілька типів
        print("Дані не знайдено")

    except ZeroDivisionError:
        print("Ділення на нуль")

    else:
        print(f"Результат: {result:.2f}")   # тільки якщо try пройшов без винятку

    finally:
        print("Спробу зафіксовано.")        # завжди


for demo in ["500", "-10", "0"]:
    process(demo)


Результат: 2.00
Спробу зафіксовано.
Помилка значення: Ціна -10.0 не може бути від'ємною
Спробу зафіксовано.
Ділення на нуль
Спробу зафіксовано.


## Далі

Урок 14 («Файли I/O, менеджери контексту, JSON») природно продовжує цю тему: контекстні менеджери (`with`), з якими ти вже познайомився через `open()`, отримають повне пояснення — чому вони гарантують закриття ресурсу навіть при винятку.
